# Ingest Contract Parsing Data

This notebook ingests data from `lex_glue` (config: `ledgar`) and `theatticusproject/maud` (Merger Agreement Understanding Dataset) into a Pinecone vector database.

In [8]:
!pip uninstall pincone-client

In [1]:
!pip install datasets pinecone sentence-transformers langchain-community langchain-huggingface python-dotenv

In [6]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import warnings

warnings.filterwarnings('ignore')

## 1. Setup Pinecone & Embeddings

In [2]:
load_dotenv("../.env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "contract-parsing-index" # Hardcoded to avoid conflicting with the main app's index (lrr)
PINECONE_ENV = os.getenv("PINECONE_ENVIRONMENT", "us-east-1")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    print(f"Creating index {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1024, # mxbai-embed-large dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_ENV)
    )

index = pc.Index(INDEX_NAME)
print("Pinecone Index ready.")

Creating index contract-parsing-index...
Pinecone Index ready.


In [3]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")


## 2. Ingest `lex_glue` (Config: `ledgar`)
The `lex_glue` benchmark includes the `ledgar` dataset for contract provision classification.

In [4]:
print("Loading lex_glue ledgar dataset...")
# Using a subset of train split for demonstration. Remove '[:1000]' to load the full dataset.
ledgar = load_dataset("lex_glue", "ledgar", split="train") 
print(f"Loaded {len(ledgar)} samples.")

Loading lex_glue ledgar dataset...
Loaded 60000 samples.


In [13]:
# Create chunks of 4000 chars, with 400 chars of overlap to prevent cutting off context mid-sentence
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len
)


In [14]:
def process_ledgar(dataset, batch_size=100):
    vectors_to_upsert = []
    for i, item in enumerate(dataset):
        text = item["text"]
        label = str(item["label"])
        
        # 1. Split the text into manageable chunks
        chunks = text_splitter.split_text(text)
        
        # 2. Process each chunk
        for chunk_idx, chunk_text in enumerate(chunks):
            embedding = embeddings.embed_query(chunk_text)
            
            metadata = {
                "source": "lex_glue_ledgar",
                "label": label,
                "text": chunk_text # Store the specific chunk's text
            }
            
            # Use unique ID per chunk
            vectors_to_upsert.append((f"ledgar_{i}_{chunk_idx}", embedding, metadata))
            
            if len(vectors_to_upsert) >= batch_size:
                index.upsert(vectors=vectors_to_upsert)
                vectors_to_upsert = []
            
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
    print("Finished ingesting ledgar.")


In [ ]:
process_ledgar(ledgar)

## 3. Ingest `theatticusproject/maud` (Merger Agreement Understanding Dataset)

In [11]:
print("Loading MAUD dataset...")
maud = load_dataset("theatticusproject/maud", split="train")
print(f"Loaded {len(maud)} samples.")

Loading MAUD dataset...


Generating test split: 100%|██████████| 6651/6651 [00:00<00:00, 37153.38 examples/s]

Loaded 25827 samples.


In [ ]:
def process_maud(dataset, batch_size=100):
    vectors_to_upsert = []
    for i, item in enumerate(dataset):
        text = item.get("text", "")
        if not text:
            text = item.get("context", "")
        if not text:
            text = " ".join([str(v) for k,v in item.items() if isinstance(v, str)])
            
        # 1. Split the text into manageable chunks
        chunks = text_splitter.split_text(text)
        
        # 2. Process each chunk
        for chunk_idx, chunk_text in enumerate(chunks):
            embedding = embeddings.embed_query(chunk_text)
            
            metadata = {
                "source": "maud",
                "text": chunk_text # Store the specific chunk's text
            }
            
            # Use unique ID per chunk
            vectors_to_upsert.append((f"maud_{i}_{chunk_idx}", embedding, metadata))
            
            if len(vectors_to_upsert) >= batch_size:
                index.upsert(vectors=vectors_to_upsert)
                vectors_to_upsert = []
            
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
    print("Finished ingesting MAUD.")

process_maud(maud)

Finished ingesting MAUD.
